# Mini Project 6: Spam Email Detection Using Machine Learning

**Developer:** Daniel Richard Xavier  
**Subject:** Machine Learning & Natural Language Processing Laboratory  
**GitHub Repository:** [https://github.com/drx2k06/miniproject6](https://github.com/drx2k06/miniproject6)  

---

## 1. Project Abstract & Objectives
This study investigates machine learning techniques for email spam classification. We develop a supervised text classification pipeline utilizing:
1. **Apache SpamAssassin Public Corpus** (genuine email data containing headers and body text)
2. **TF-IDF Feature Representation** (capturing word unigrams and bigrams)
3. **Multinomial Naive Bayes (MNB)** as the primary probabilistic baseline
4. **Logistic Regression (LR)** as the discriminative linear comparison model
5. Systematic evaluation focusing on accuracy, precision, recall, F1-score, and ROC-AUC analysis.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
print("Environment initialized successfully.")

## 2. Dataset Loading & Inspection
We load the preprocessed email dataset derived from the SpamAssassin public corpus.

In [ ]:
data_path = "../data/processed/clean_emails.csv"
if not os.path.exists(data_path):
    data_path = "../data/processed/emails.csv"

df = pd.read_csv(data_path)
print(f"Total unique emails loaded: {len(df):,}")
print("\nClass Distribution:")
print(df["label"].value_counts())
print("\nClass Proportions:")
print(df["label"].value_counts(normalize=True).mul(100).round(2).astype(str) + "%")
df.head(3)

## 3. Exploratory Data Analysis: Email Length Distribution
Analyzing message character counts between Ham and Spam emails.

In [ ]:
df["char_count"] = df["text"].astype(str).str.len()

plt.figure(figsize=(9, 4))
sns.boxplot(data=df, x="label", y="char_count", palette=["#2b5c8f", "#d95f02"])
plt.yscale("log")
plt.title("Character Count Distribution by Class (Log Scale)", fontsize=12, fontweight="bold")
plt.xlabel("Class")
plt.ylabel("Character Length (Log)")
plt.show()

## 4. Stratified Train/Test Split (80/20)
To prevent data leakage, vectorizers and models are trained strictly on the training partition.

In [ ]:
X = df["cleaned_text"] if "cleaned_text" in df.columns else df["text"]
y = df["label_num"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Training set: {len(X_train)} samples (Ham: {(y_train == 0).sum()}, Spam: {(y_train == 1).sum()})")
print(f"Test set:     {len(X_test)} samples (Ham: {(y_test == 0).sum()}, Spam: {(y_test == 1).sum()})")

## 5. Model Pipeline Construction & Training
We build scikit-learn Pipelines combining `TfidfVectorizer` (unigrams + bigrams) with:
1. **Multinomial Naive Bayes** (`alpha=0.1`)
2. **Logistic Regression** (`C=1.0`, `solver='liblinear'`)

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000,
    sublinear_tf=True,
    stop_words="english"
)

pipe_nb = Pipeline([("tfidf", vectorizer), ("classifier", MultinomialNB(alpha=0.1))])
pipe_lr = Pipeline([("tfidf", vectorizer), ("classifier", LogisticRegression(C=1.0, max_iter=1000, solver="liblinear", random_state=42))])

# Fit pipelines
print("Training Multinomial Naive Bayes...")
pipe_nb.fit(X_train, y_train)

print("Training Logistic Regression...")
pipe_lr.fit(X_train, y_train)

print("Training completed.")

## 6. Evaluation on Unseen Test Partition

In [ ]:
models = {"Multinomial Naive Bayes": pipe_nb, "Logistic Regression": pipe_lr}
comparison = []

for name, model in models.items():
    preds = model.predict(X_test)
    probas = model.predict_proba(X_test)[:, 1]
    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()
    
    comparison.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision (Spam)": precision_score(y_test, preds),
        "Recall (Spam)": recall_score(y_test, preds),
        "F1-Score (Spam)": f1_score(y_test, preds),
        "ROC-AUC": roc_auc_score(y_test, probas),
        "False Positives (FP)": fp,
        "False Negatives (FN)": fn
    })

comp_df = pd.DataFrame(comparison)
comp_df.style.format({
    "Accuracy": "{:.2%}",
    "Precision (Spam)": "{:.2%}",
    "Recall (Spam)": "{:.2%}",
    "F1-Score (Spam)": "{:.2%}",
    "ROC-AUC": "{:.4f}"
})

## 7. Confusion Matrix Visualization
Heatmap visualization illustrating True Negatives, False Positives, Missed Spam, and Detected Spam.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, model) in zip(axes, models.items()):
    preds = model.predict(X_test)
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Pred Ham", "Pred Spam"], yticklabels=["Actual Ham", "Actual Spam"])
    ax.set_title(f"Confusion Matrix: {name}", fontweight="bold")
    ax.set_ylabel("Actual Label")
    ax.set_xlabel("Predicted Label")

plt.tight_layout()
plt.show()

## 8. Interactive Classification Test
Predict arbitrary text samples to verify inference responsiveness.

In [ ]:
sample_test = [
    "Subject: Urgent: Verify your bank account password immediately or account will be closed!",
    "Subject: Team meeting notes and follow up action items for next Tuesday at 3pm."
]

for s in sample_test:
    p = pipe_lr.predict([s])[0]
    prob = pipe_lr.predict_proba([s])[0]
    label = "SPAM" if p == 1 else "HAM"
    print(f"\nSample: {s}")
    print(f"Prediction: {label} (Confidence: {prob[p]:.2%})")